# `extract_hints` — extraction d'indices structurels d'une question

Référence : chapitre **« Understanding the Question Before Searching »** (4ème onglet du Google Doc Faseya).

**Idée centrale** : une question utilisateur n'est pas un seul input. C'est la matière première pour DEUX préparations distinctes :

| Préparation | Pour quoi | Contenu |
|---|---|---|
| `RetrievalQuery`  | la recherche dans les documents | query nettoyée, rewrites, anchor keywords, **hints structurels** |
| `GenerationBrief` | la formulation de la réponse par le LLM | question originale, format attendu, désambiguïsation |

Cette première fonction `extract_hints(question)` couvre les **hints structurels** : page, section, layout, document. Elle sert à **filtrer la recherche en amont** : si l'utilisateur dit *« c'est dans le tableau page 3 »*, on cherche d'abord dans les pages où `has_vector_table=True` et `page_num=3` (cf. `page_df` produit par `parse_pdf`).

| Hint | Exemples détectés |
|---|---|
| `page_hint`    | `page 1`, `p. 12`, `à la page 47`, `on page 5` |
| `section_hint` | `dans la section exclusions`, `section called Limits`, `in the section Risks` |
| `layout_hint`  | `tableau` / `table` → `table` ; `figure` / `image` → `image` ; `en-tête` → `header` ; `pied de page` → `footer` |
| `document_hint`| `dernière version`, `latest version`, `dans le contrat X` |

Approche : regex déterministe (rapide, testable, pas de coût LLM). Un appel LLM optionnel sera ajouté pour les cas flous quand on aura les vraies questions de Kezhan.

In [1]:
# À exécuter une seule fois : installer le package en mode editable
import sys
!{sys.executable} -m pip install -q -e ../../../..


[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip
ERROR: file:///e:/Project/DocProcessing does not appear to be a Python project: neither 'setup.py' nor 'pyproject.toml' found.


In [2]:
import json
from dataclasses import asdict
from docpipeline.question.understand_question import extract_hints

# Banque de questions exemples (à remplacer par celles de Kezhan)
QUESTIONS = [
    "Quelle est la date d'effet du contrat ? C'est en page 1.",
    "Look in the exclusions section for the flooding clause.",
    "What's the limit per claim? It's in the recap table near the end.",
    "Dans la dernière version du contrat, quel est le plafond ?",
    "Find the policy number — it's usually in the header.",
    "Combien coûte l'assurance ?",   # aucun hint
]

for q in QUESTIONS:
    hints = extract_hints(q)
    print(f"Q: {q}")
    print(f"   → {json.dumps(asdict(hints), ensure_ascii=False)}")
    print()

Q: Quelle est la date d'effet du contrat ? C'est en page 1.
   → {"section_hint": null, "page_hint": 1, "layout_hint": null, "document_hint": null, "raw_signals": ["regex_page:'page 1'"]}

Q: Look in the exclusions section for the flooding clause.
   → {"section_hint": null, "page_hint": null, "layout_hint": null, "document_hint": null, "raw_signals": []}

Q: What's the limit per claim? It's in the recap table near the end.
   → {"section_hint": null, "page_hint": null, "layout_hint": "table", "document_hint": null, "raw_signals": ["regex_layout:table→table"]}

Q: Dans la dernière version du contrat, quel est le plafond ?
   → {"section_hint": null, "page_hint": null, "layout_hint": null, "document_hint": "la dernière version du contrat", "raw_signals": ["regex_document:'la dernière version du contrat'"]}

Q: Find the policy number — it's usually in the header.
   → {"section_hint": null, "page_hint": null, "layout_hint": "header", "document_hint": null, "raw_signals": ["regex_layout:h